In [4]:
# === CONFIGURAÇÃO ===
# Escolha a empresa para o relatório:
# Opções: 'predileto', 'logika', 'mosaic', 'ricker', 'sighir', 'manchur', 'felka', 'bellavia', 'klabin', 'thomaz', 'scania', 'zirix'
# Use None para relatório geral (todas as empresas)

EMPRESA = 'predileto'

In [5]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import os
import tempfile

from reportlab.lib.pagesizes import A4
from reportlab.lib.colors import HexColor
from reportlab.lib.units import mm
from reportlab.pdfgen import canvas


class ReportGenerator:

    def __init__(self, csvPath, outputPath, companyFilter=None):
        self.csvPath = csvPath
        self.outputPath = outputPath
        self.companyFilter = companyFilter
        self.reportDate = datetime.now()
        self.pageWidth, self.pageHeight = A4
        self.margin = 18 * mm
        self.contentWidth = self.pageWidth - 2 * self.margin

        self.darkBlue = HexColor('#1B4F72')
        self.mixBadge = HexColor('#3498DB')
        self.suntechBadge = HexColor('#1ABC9C')
        self.white = HexColor('#FFFFFF')
        self.mediumGray = HexColor('#BDC3C7')
        self.darkGray = HexColor('#2C3E50')
        self.cardBorder = HexColor('#D5D8DC')
        self.tableHeaderBg = HexColor('#1B4F72')
        self.tableRowAlt = HexColor('#F8F9FA')
        self.tempDir = tempfile.mkdtemp()

    def loadData(self):
        self.dfAll = pd.read_csv(self.csvPath)
        self.dfAll['installation_date'] = pd.to_datetime(self.dfAll['installation_date'], errors='coerce')
        self.dfAll['timestamp'] = pd.to_datetime(self.dfAll['timestamp'], errors='coerce')
        self.dfAll['updated_at'] = pd.to_datetime(self.dfAll['updated_at'], errors='coerce')

        if self.companyFilter:
            self.df = self.dfAll[
                (self.dfAll['company'] == self.companyFilter) &
                (self.dfAll['installed'] == True) &
                (~self.dfAll['plate'].str.lower().str.contains('teste|maleta|bancada|desativado', na=False))
            ].copy()
            self.companyLabel = self.companyFilter.upper()
        else:
            self.df = self.dfAll[self.dfAll['installed'] == True].copy()
            self.companyLabel = 'GERAL'

    def calculateMetrics(self):
        self.totalInstalled = len(self.df)
        self.baseMix = int((self.df['telemetry'] == 'mix').sum())
        self.baseSuntech = int((self.df['telemetry'] == 'suntech').sum())

        oneMonthAgo = self.reportDate - pd.Timedelta(days=30)
        self.recentCount = int((self.df['installation_date'] >= oneMonthAgo).sum())

        if self.totalInstalled > 0:
            self.mixPercent = round(self.baseMix / self.totalInstalled * 100, 1)
            self.suntechPercent = round(self.baseSuntech / self.totalInstalled * 100, 1)
        else:
            self.mixPercent = 0
            self.suntechPercent = 0

        tableData = self.df.sort_values('installation_date', ascending=False)
        self.tableData = tableData[['plate', 'installation_date', 'telemetry', 'company']].copy()
        self.tableData = self.tableData.dropna(subset=['installation_date'])

    def generateCumulativeChart(self):
        data = self.df.dropna(subset=['installation_date']).copy()
        data = data.sort_values('installation_date')
        data['month'] = data['installation_date'].dt.to_period('M')
        monthly = data.groupby('month').size().cumsum()

        fig, ax = plt.subplots(figsize=(5.5, 3.2))
        dates = [p.to_timestamp() for p in monthly.index]
        values = monthly.values

        ax.plot(dates, values, color='#2980B9', linewidth=2.5, marker='o', markersize=4, zorder=3)
        ax.fill_between(dates, values, alpha=0.15, color='#2980B9')
        ax.set_ylabel('TOTAL ACUMULADO', fontsize=8, fontweight='bold', color='#555555')

        if len(dates) >= 2:
            ax.set_xlabel(f'DATA ({dates[0].strftime("%b %Y")} - {dates[-1].strftime("%b %Y")})',
                          fontsize=7, color='#555555')

        ax.tick_params(axis='both', labelsize=7, colors='#555555')
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()

        chartPath = os.path.join(self.tempDir, 'cumulative.png')
        plt.savefig(chartPath, dpi=200, bbox_inches='tight', transparent=True)
        plt.close()
        return chartPath

    def generateDonutChart(self):
        fig, ax = plt.subplots(figsize=(3, 3))
        sizes = [self.mixPercent, self.suntechPercent]
        colors = ['#2C3E50', '#48C9B0']

        wedges, _ = ax.pie(sizes, colors=colors, startangle=90,
                           wedgeprops=dict(width=0.35, edgecolor='white', linewidth=2))

        labels = [f'{self.mixPercent}%\nMix', f'{self.suntechPercent}%\nSuntech']
        for i, (wedge, label) in enumerate(zip(wedges, labels)):
            ang = (wedge.theta2 + wedge.theta1) / 2
            radius = 1.35
            x = radius * np.cos(np.radians(ang))
            y = radius * np.sin(np.radians(ang))
            ha = 'left' if x > 0 else 'right'
            ax.text(x, y, label, ha=ha, va='center', fontsize=9, fontweight='bold', color=colors[i])

        ax.axis('equal')
        plt.tight_layout()

        chartPath = os.path.join(self.tempDir, 'donut.png')
        plt.savefig(chartPath, dpi=200, bbox_inches='tight', transparent=True)
        plt.close()
        return chartPath

    def drawRoundedRect(self, c, x, y, w, h, radius, fillColor=None, strokeColor=None, strokeWidth=0.5):
        c.saveState()
        if fillColor:
            c.setFillColor(fillColor)
        if strokeColor:
            c.setStrokeColor(strokeColor)
            c.setLineWidth(strokeWidth)
        p = c.beginPath()
        p.roundRect(x, y, w, h, radius)
        p.close()
        if fillColor and strokeColor:
            c.drawPath(p, fill=1, stroke=1)
        elif fillColor:
            c.drawPath(p, fill=1, stroke=0)
        elif strokeColor:
            c.drawPath(p, fill=0, stroke=1)
        c.restoreState()

    def drawBadge(self, c, x, y, text, color, width=50, height=16):
        self.drawRoundedRect(c, x, y, width, height, height / 2, fillColor=color)
        c.saveState()
        c.setFillColor(self.white)
        c.setFont('Helvetica-Bold', 7)
        c.drawCentredString(x + width / 2, y + 4.5, text)
        c.restoreState()

    def drawHeader(self, c):
        y = self.pageHeight - 32 * mm
        self.drawRoundedRect(c, self.margin, y, self.contentWidth, 22 * mm, 3 * mm, fillColor=self.darkBlue)

        c.saveState()
        c.setFillColor(self.white)
        c.setFont('Helvetica-Bold', 8)
        c.drawString(self.margin + 6 * mm, y + 16 * mm, u'RELATÓRIO DE OPERAÇÕES & TELEMETRIA')

        monthNames = ['JAN', 'FEV', 'MAR', 'ABR', 'MAI', 'JUN', 'JUL', 'AGO', 'SET', 'OUT', 'NOV', 'DEZ']
        dateStr = f'{self.reportDate.day} {monthNames[self.reportDate.month - 1]} {self.reportDate.year}'
        c.setFont('Helvetica', 8)
        c.drawRightString(self.margin + self.contentWidth - 6 * mm, y + 16 * mm, dateStr)
        c.restoreState()

        y2 = y - 3 * mm
        c.saveState()
        logoX = self.margin + 6 * mm
        logoY = y2 - 10 * mm

        c.setFillColor(self.darkBlue)
        c.circle(logoX + 5 * mm, logoY + 4 * mm, 5.5 * mm, fill=1, stroke=0)
        c.setFillColor(self.white)
        c.setFont('Helvetica-Bold', 13)
        c.drawCentredString(logoX + 5 * mm, logoY + 1 * mm, self.companyLabel[0])

        c.setFillColor(self.darkBlue)
        c.setFont('Helvetica-Bold', 14)
        c.drawString(logoX + 14 * mm, logoY + 1 * mm, self.companyLabel)

        c.setFillColor(HexColor('#666666'))
        c.setFont('Helvetica', 6.5)
        subtitleX = self.margin + self.contentWidth - 6 * mm
        c.drawRightString(subtitleX, logoY + 6 * mm, u'Status de instalação de telemetria e')
        c.drawRightString(subtitleX, logoY + 1 * mm, u'etilômetros na frota ativa, excluindo testes.')
        c.restoreState()
        return y2 - 18 * mm

    def drawMetricCards(self, c, startY):
        cardWidth = (self.contentWidth - 2 * 4 * mm) / 3
        cardHeight = 26 * mm
        y = startY

        cards = [
            {'title': 'TOTAL INSTALADO', 'subtitle': u'(Veículos Ativos)',
             'value': str(self.totalInstalled), 'extra': f'+{self.recentCount} no último mês',
             'badge': None, 'badgeColor': None},
            {'title': 'BASE MIX', 'subtitle': u'(Etilômetros)',
             'value': str(self.baseMix), 'extra': None,
             'badge': 'Mix', 'badgeColor': self.mixBadge},
            {'title': 'BASE SUNTECH', 'subtitle': '',
             'value': str(self.baseSuntech), 'extra': None,
             'badge': 'Suntech', 'badgeColor': self.suntechBadge},
        ]

        for i, card in enumerate(cards):
            x = self.margin + i * (cardWidth + 4 * mm)
            self.drawRoundedRect(c, x, y - cardHeight, cardWidth, cardHeight, 3 * mm,
                                 fillColor=self.white, strokeColor=self.cardBorder, strokeWidth=0.8)
            c.saveState()
            c.setFillColor(self.darkBlue)
            c.setFont('Helvetica-Bold', 7.5)
            c.drawString(x + 5 * mm, y - 7 * mm, card['title'])
            if card['subtitle']:
                c.setFont('Helvetica', 5.5)
                c.setFillColor(HexColor('#888888'))
                c.drawString(x + 5 * mm, y - 11 * mm, card['subtitle'])
            c.setFillColor(self.darkBlue)
            c.setFont('Helvetica-Bold', 26)
            c.drawString(x + 5 * mm, y - cardHeight + 4 * mm, card['value'])
            if card['badge']:
                valueWidth = c.stringWidth(card['value'], 'Helvetica-Bold', 26)
                self.drawBadge(c, x + 5 * mm + valueWidth + 4 * mm, y - cardHeight + 5 * mm,
                               card['badge'], card['badgeColor'])
            if card['extra']:
                c.setFont('Helvetica', 5.5)
                c.setFillColor(self.suntechBadge)
                c.drawString(x + 5 * mm, y - cardHeight + 13 * mm, card['extra'])
            c.restoreState()

        return y - cardHeight - 4 * mm

    def drawCharts(self, c, startY):
        c.saveState()
        chartHeight = 50 * mm
        leftW = self.contentWidth * 0.58
        rightW = self.contentWidth * 0.38
        gap = self.contentWidth * 0.04

        c.setFont('Helvetica-Bold', 8)
        c.setFillColor(self.darkGray)
        c.drawCentredString(self.margin + leftW / 2, startY, u'EVOLUÇÃO CUMULATIVA DE INSTALAÇÕES')
        c.drawCentredString(self.margin + leftW + gap + rightW / 2, startY, u'PROPORÇÃO POR')
        c.drawCentredString(self.margin + leftW + gap + rightW / 2, startY - 4 * mm, 'TELEMETRIA')
        c.restoreState()

        cumulativePath = self.generateCumulativeChart()
        chartY = startY - 6 * mm - chartHeight
        c.drawImage(cumulativePath, self.margin, chartY, width=leftW, height=chartHeight,
                    preserveAspectRatio=True, anchor='sw')

        donutPath = self.generateDonutChart()
        c.drawImage(donutPath, self.margin + leftW + gap, chartY, width=rightW, height=chartHeight,
                    preserveAspectRatio=True, anchor='sw')
        return chartY - 4 * mm

    def drawSectionTitle(self, c, y):
        c.saveState()
        c.setStrokeColor(self.mediumGray)
        c.setLineWidth(0.5)
        c.line(self.margin, y + 2 * mm, self.margin + self.contentWidth, y + 2 * mm)
        c.setFont('Helvetica-Bold', 9)
        c.setFillColor(self.darkGray)
        c.drawCentredString(self.pageWidth / 2, y - 6 * mm,
                            u'AMOSTRA RECENTE: ÚLTIMOS VEÍCULOS ATUALIZADOS')
        c.restoreState()
        return y - 12 * mm

    def drawTableHeader(self, c, y):
        rowHeight = 8 * mm
        self.colWidths = [self.contentWidth * 0.2, self.contentWidth * 0.3,
                          self.contentWidth * 0.25, self.contentWidth * 0.25]
        headers = ['PLACA', u'DATA INSTALAÇÃO', 'MODELO', 'EMPRESA']

        self.drawRoundedRect(c, self.margin, y - rowHeight, self.contentWidth, rowHeight,
                             2 * mm, fillColor=self.tableHeaderBg)
        c.saveState()
        c.setFillColor(self.white)
        c.setFont('Helvetica-Bold', 7)
        x = self.margin
        for i, header in enumerate(headers):
            c.drawCentredString(x + self.colWidths[i] / 2, y - rowHeight + 2.5 * mm, header)
            x += self.colWidths[i]
        c.restoreState()
        return y - rowHeight

    def drawTableRows(self, c, startY, startIdx=0):
        rowHeight = 7 * mm
        minY = 28 * mm
        y = startY
        rowsDrawn = 0

        for idx in range(startIdx, len(self.tableData)):
            if y - rowHeight < minY:
                break
            row = self.tableData.iloc[idx]

            if idx % 2 == 0:
                c.saveState()
                c.setFillColor(self.tableRowAlt)
                c.rect(self.margin, y - rowHeight, self.contentWidth, rowHeight, fill=1, stroke=0)
                c.restoreState()

            c.saveState()
            c.setStrokeColor(HexColor('#E8E8E8'))
            c.setLineWidth(0.3)
            c.line(self.margin, y - rowHeight, self.margin + self.contentWidth, y - rowHeight)
            c.restoreState()

            x = self.margin
            c.saveState()
            c.setFillColor(self.darkGray)
            c.setFont('Helvetica', 7)
            plate = str(row['plate']) if pd.notna(row['plate']) else ''
            c.drawCentredString(x + self.colWidths[0] / 2, y - rowHeight + 2.5 * mm, plate)
            x += self.colWidths[0]

            dateStr = row['installation_date'].strftime('%d/%m/%Y') if pd.notna(row['installation_date']) else ''
            c.drawCentredString(x + self.colWidths[1] / 2, y - rowHeight + 2.5 * mm, dateStr)
            x += self.colWidths[1]
            c.restoreState()

            telemetry = str(row['telemetry']).capitalize() if pd.notna(row['telemetry']) else ''
            badgeColor = self.mixBadge if row['telemetry'] == 'mix' else self.suntechBadge
            self.drawBadge(c, x + (self.colWidths[2] - 48) / 2, y - rowHeight + 2 * mm,
                           telemetry, badgeColor, 48, 14)
            x += self.colWidths[2]

            c.saveState()
            c.setFillColor(self.darkGray)
            c.setFont('Helvetica', 7)
            empresa = str(row['company']).capitalize() if pd.notna(row['company']) else ''
            c.drawCentredString(x + self.colWidths[3] / 2, y - rowHeight + 2.5 * mm, empresa)
            c.restoreState()

            y -= rowHeight
            rowsDrawn += 1

        return y, startIdx + rowsDrawn

    def drawFooter(self, c):
        y = 18 * mm
        c.saveState()
        c.setFillColor(HexColor('#F5F5F5'))
        c.rect(0, 0, self.pageWidth, y + 5 * mm, fill=1, stroke=0)
        c.setFillColor(HexColor('#888888'))
        c.setFont('Helvetica', 5)
        c.drawCentredString(self.pageWidth / 2, y,
                            f'{self.companyLabel} \u2022 Gerado em {self.reportDate.strftime("%d/%m/%Y %H:%M")}')
        c.drawCentredString(self.pageWidth / 2, y - 4 * mm,
                            u'Confidencialidade: as informações contidas neste relatório são sigilosas.')
        c.drawCentredString(self.pageWidth / 2, y - 8 * mm,
                            u'Gerado por Sistema Interno \u2022 Todos os direitos reservados \u2022 v2.1')
        c.restoreState()

    def generate(self):
        self.loadData()
        self.calculateMetrics()

        c = canvas.Canvas(self.outputPath, pagesize=A4)
        y = self.drawHeader(c)
        y = self.drawMetricCards(c, y)
        y = self.drawCharts(c, y)
        y = self.drawSectionTitle(c, y)
        y = self.drawTableHeader(c, y)
        y, nextIdx = self.drawTableRows(c, y)
        self.drawFooter(c)
        c.showPage()

        while nextIdx < len(self.tableData):
            c.saveState()
            c.setFont('Helvetica-Bold', 9)
            c.setFillColor(self.darkGray)
            y = self.pageHeight - 20 * mm
            c.drawCentredString(self.pageWidth / 2, y,
                                u'AMOSTRA RECENTE: ÚLTIMOS VEÍCULOS ATUALIZADOS (continuação)')
            c.restoreState()
            y -= 5 * mm
            y = self.drawTableHeader(c, y)
            y, nextIdx = self.drawTableRows(c, y, nextIdx)
            self.drawFooter(c)
            c.showPage()

        c.save()
        return self.outputPath


In [6]:
generator = ReportGenerator('DataBase.csv', 'output/report.pdf', companyFilter=EMPRESA)
outputPath = generator.generate()
print(f'Relatório gerado: {outputPath}')
print(f'Empresa: {EMPRESA or "Todas"}')
print(f'Total instalado: {generator.totalInstalled}')
print(f'Base Mix: {generator.baseMix} ({generator.mixPercent}%)')
print(f'Base Suntech: {generator.baseSuntech} ({generator.suntechPercent}%)')

Relatório gerado: output/report.pdf
Empresa: predileto
Total instalado: 60
Base Mix: 48 (80.0%)
Base Suntech: 12 (20.0%)
